BAR Model Vignette
This notebook demonstrates how to use the BAR class with AR(3) synthetic data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
print("PyMC version:", pm.__version__)

# Reload
import importlib
import bar
importlib.reload(bar)
from bar import BAR

1. Simulate AR(3) Data

In [ ]:
# Simulate AR(3) data
y = [0, 0, 0]
phi_true = [0.5, -0.3, 0.2]
sigma_true = 1.0
np.random.seed(2102)
for t in range(3, 100):
    y_t = phi_true[0]*y[-1] + phi_true[1]*y[-2] + phi_true[2]*y[-3] + np.random.normal(0, sigma_true)
    y.append(y_t)
y = np.array(y)

plt.figure(figsize=(10, 4))
plt.plot(y)
plt.title("Simulated AR(3) Time Series")
plt.xlabel("Time")
plt.ylabel("Value")
plt.grid()
plt.tight_layout()
plt.show()


2. Create and Fit the BAR Model

In [ ]:
bar = BAR(p=3)
bar.fit(y)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [phi, sigma]
Output()
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 23 seconds.
2.1 Refit the Model with Custom Priors
Here we re-initialize and re-fit the BAR model using Laplace priors on the AR coefficients and a HalfCauchy prior on the noise standard deviation.

In [ ]:
custom_priors = {
    'phi': lambda shape: pm.Laplace("phi", mu=0, b=0.5, shape=shape),
    'sigma': lambda: pm.HalfCauchy("sigma", beta=1)
}
bar.priors = custom_priors
bar.fit(y)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [phi, sigma]
Output()
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 22 seconds.

In [ ]:
bar.summary()

3. Plot the Input Time Series

In [ ]:

bar.plot_series()

4. Visualize Prior Distributions

In [ ]:
# bar = BAR(p=3) # Note: Can invoke plot_priors() before fit()!
bar.plot_priors()

5. Summarize Posterior Distributions

In [ ]:

bar.summary()

In [ ]:
6. Plot MCMC Trace

7. Forecast Future Values

In [ ]:
forecast_mean, lower, upper, forecasts = bar.forecast(steps=10)
print("Forecast mean:", forecast_mean)

8. Plot Forecast with Credible Intervals

In [ ]:
bar.plot_forecast(steps=10)